In [0]:
from pyspark.sql import functions
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
 
df = spark.read.table("project_apis.bronze.districts_ibge_raw")

df_schema = ArrayType(
    StructType([
        StructField("id", StringType(), True),
        StructField("nome", StringType(), True),
        StructField("municipio", StructType([
            StructField("nome", StringType(), True),
            StructField("microrregiao", StructType([
                StructField("nome", StringType(), True),
                StructField("mesorregiao", StructType([
                    StructField("nome", StringType(), True),
                    StructField("UF", StructType([
                        StructField("nome", StringType(), True),
                        StructField("sigla", StringType(), True),
                        StructField("regiao", StructType([
                            StructField("nome", StringType(), True),
                            StructField("sigla", StringType(), True),
                        ]), True)
                    ]), True)
                ]), True)
            ]), True)
        ]), True)
    ])
)

print(df_schema)



In [0]:
df1 = df.withColumn("distritos_array",functions.from_json(functions.col("raw_json"), df_schema))
df1.show()
print(df1)
df1.printSchema

In [0]:
df2 = df1.withColumn("distritos", functions.explode(functions.col("distritos_array")))
#df2.show()
df2.printSchema()

df_split = df2.select(
    functions.col("distritos.id").alias("id"),
    functions.col("distritos.nome").alias("nome"),
    functions.col("distritos.municipio.nome").alias("nome_municipio"),
    functions.col("distritos.municipio.microrregiao.nome").alias("nome_microrregiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.nome").alias("nome_mesorregiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.nome").alias("nome_UF"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.sigla").alias("sigla_UF"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.regiao.nome").alias("nome_regiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.regiao.sigla").alias("sigla_regiao"),
    functions.col("ingestion_timestamp"),
    functions.col("source")
)

df_split.show()
#df_split.printSchema()
